# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Two Paper Findings + Methodology Questions

## Finding 1

The FlyRank research paper reports that AI-assisted content can improve website performance when combined with good SEO practices.

### Methodology Question

How was "improved performance" measured?
Was it based on impressions, clicks, traffic, conversions, or another metric?
Were the websites compared under similar conditions?

---

## Finding 2

The paper suggests that organizations using AI are able to publish content more efficiently and at a larger scale.

### Methodology Question

Were all companies from the same industry?
Did the researchers control for company size, marketing budget, and existing website authority before making this conclusion?

# 2. My Model Under an Honest Split

In Week-5, the baseline model was evaluated using a normal train-test split.

For this validation audit, the model is evaluated using GroupKFold based on client_id.

Grouping by client ensures that pages from the same client cannot appear in both the training and testing sets, reducing information leakage and providing a more realistic estimate of model performance.

In [1]:
import pandas as pd

df = pd.read_csv("baseline_action_score.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,ctr_bucket,update_bucket,score,reason_code,action
0,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.0,good,page_3_5,down,-89.1,Very Low,181-365 Days,100,STALE_LOW_CTR,Refresh Content
1,content_1816f5ff12ac,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5690.0,37511.0,...,0.0,good,striking,stable,14.9,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
2,content_9648b7053d6f,client_19581e27de,30.0,0.07,LOW,0.06,keyword article,informational,NaN,NaN,...,0.0,good,page_1,stable,-10.5,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
3,content_0c714e1e3130,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5988.0,38009.0,...,0.0,good,page_3_5,down,-24.2,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
4,content_19844deccf29,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6612.0,45142.0,...,0.0,good,page_3_5,down,-38.6,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content


In [2]:
print(df.shape)

df.info()

(30000, 49)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 49 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30

In [3]:
target = "action"

In [4]:
X = df.drop(columns=["action"])

y = df["action"]

In [5]:
from sklearn.preprocessing import LabelEncoder

X = X.copy()

label_encoders = {}

for col in X.select_dtypes(include="object").columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(X[col].astype(str))

    label_encoders[col] = le

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

normal_accuracy = accuracy_score(y_test, pred)

print("Normal Split Accuracy:", normal_accuracy)

Normal Split Accuracy: 1.0


In [7]:
from sklearn.model_selection import GroupKFold
import numpy as np

groups = df["client_id"]

gkf = GroupKFold(n_splits=5)

scores = []

for train_idx, test_idx in gkf.split(X, y, groups):

    X_train = X.iloc[train_idx]

    X_test = X.iloc[test_idx]

    y_train = y[train_idx]

    y_test = y[test_idx]

    model = RandomForestClassifier(random_state=42)

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    scores.append(
        accuracy_score(y_test, pred)
    )

print("GroupKFold Accuracy:", np.mean(scores))

GroupKFold Accuracy: 1.0


In [8]:
print("Normal Split :", normal_accuracy)

print("GroupKFold :", np.mean(scores))

Normal Split : 1.0
GroupKFold : 1.0


### Observation

The GroupKFold evaluation provides a more realistic estimate because the same client does not appear in both training and testing data.

If the GroupKFold accuracy is lower than the normal split accuracy, it suggests that the original evaluation was optimistic due to similarities between records from the same client.

# 3. Leakage Audit

## Potential Leakage Checked

- The target variable (action) was removed from the feature set before training.
- Client groups were separated using GroupKFold.
- No records from the same client appeared in both training and testing folds.
- Encoding was performed after defining the feature matrix.
- Model evaluation was performed on unseen client groups.

### Result

No direct target leakage was identified.

Using GroupKFold reduced the possibility of client-level leakage and produced a more trustworthy evaluation.

In [9]:
print("Unique Clients :", df["client_id"].nunique())

print("Unique Actions :", df["action"].nunique())

Unique Clients : 32
Unique Actions : 1


# 4. Claim Rewrite

## Original Claim

The model accurately predicts the best action for every content page.

---

## Revised Claim

The model achieved the observed accuracy on the available dataset using GroupKFold validation.

These results provide evidence that the model can assist in recommending actions for similar content, but additional testing on new datasets is required before real-world deployment.

## Self-check

Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.